# Pulse of Prevention — Heart Health Data Analysis

## Python / Google Colab End-to-End EDA

**Company:** HealthPulse Analytics

**Objective:** Analyze patient heart-health data to understand demographic and clinical patterns, identify factors associated with heart disease, profile higher-risk groups, and support preventive decision-making.

### Workflow
1. Upload the dataset
2. Inspect the data
3. Clean and preprocess
4. Basic EDA — 10 questions
5. Medium EDA — 10 questions
6. Advanced analysis — 5 questions
7. Key findings and recommendations
8. Save the cleaned dataset

**Important source note:** The supplied document contains unrelated healthcare/loan wording in its final "Desired Outcome" and some additional considerations. This notebook follows the actual heart-health case study, dataset, data dictionary, questions, and stated objective.


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from IPython.display import display
from scipy.stats import ttest_ind
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


## 2. Upload the Dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))

print("Uploaded file:", file_name)


## 3. Load the Dataset

In [ ]:
data = pd.read_csv(file_name)

print("Dataset loaded successfully.")
print("Rows:", data.shape[0])
print("Columns:", data.shape[1])

display(data.head())


## 4. Initial Data Inspection

In [ ]:
print("Shape:", data.shape)

print("\nColumns:")
print(data.columns.tolist())

print("\nData types:")
display(data.dtypes.to_frame("Data Type"))

print("\nSummary statistics:")
display(data.describe().T)


## 5. Check Missing Values

In [ ]:
missing = pd.DataFrame({
    "Missing Values": data.isna().sum(),
    "Missing %": (data.isna().mean() * 100).round(2)
}).sort_values("Missing Values", ascending=False)

display(missing)


## 6. Check Duplicate Rows

In [ ]:
duplicate_count = data.duplicated().sum()
print("Duplicate rows:", duplicate_count)


## 7. Check Unique Values and Data Quality

In [ ]:
for col in data.columns:
    print(f"\n{col}:")
    print("Unique values:", sorted(data[col].dropna().unique().tolist())[:30])
    print("Number of unique values:", data[col].nunique())


## 8. Data Cleaning and Preprocessing

In [ ]:
cleaned_data = data.copy()

# Ensure numeric columns are numeric
numeric_cols = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]

for col in numeric_cols:
    cleaned_data[col] = pd.to_numeric(cleaned_data[col], errors="coerce")

# Remove rows with missing values if any are present
before = len(cleaned_data)
cleaned_data = cleaned_data.dropna().reset_index(drop=True)
removed_missing = before - len(cleaned_data)

# Remove exact duplicates
before_dup = len(cleaned_data)
cleaned_data = cleaned_data.drop_duplicates().reset_index(drop=True)
removed_duplicates = before_dup - len(cleaned_data)

print("Rows removed due to missing values:", removed_missing)
print("Duplicate rows removed:", removed_duplicates)
print("Final rows:", len(cleaned_data))


## 9. Outlier Detection — Clinical Measurements

In [ ]:
clinical_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]

outlier_summary = []

for col in clinical_cols:
    q1 = cleaned_data[col].quantile(0.25)
    q3 = cleaned_data[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = ((cleaned_data[col] < lower) | (cleaned_data[col] > upper)).sum()
    outlier_summary.append([col, q1, q3, lower, upper, int(count)])

outlier_df = pd.DataFrame(
    outlier_summary,
    columns=["Feature","Q1","Q3","Lower Bound","Upper Bound","Outlier Count"]
)

display(outlier_df)

# Keep clinical observations for analysis rather than deleting valid medical extremes automatically.
print("Note: Potential outliers are identified but not automatically removed because extreme clinical values can be meaningful observations.")


## 10. Normalized Clinical Measurements

In [ ]:
# Standardized versions for multivariate analysis/modeling.
normalized_data = cleaned_data.copy()

for col in ["trestbps", "chol", "thalach", "oldpeak"]:
    mean = normalized_data[col].mean()
    std = normalized_data[col].std()
    normalized_data[col + "_z"] = (normalized_data[col] - mean) / std

display(normalized_data.head())


## 11. Save Cleaned Dataset

In [ ]:
cleaned_file = "heart_health_cleaned.csv"
cleaned_data.to_csv(cleaned_file, index=False)

print("Saved:", cleaned_file)


# BASIC LEVEL QUESTIONS

All 10 Basic questions from the supplied case study are answered below.

## Basic 1 — What is the average age of patients in the dataset?

In [ ]:
avg_age = cleaned_data["age"].mean()
print(f"Average patient age: {avg_age:.2f} years")


## Basic 2 — What is the gender distribution of patients?

In [ ]:
gender_distribution = cleaned_data["sex"].map({
    1: "Male",
    0: "Female"
}).value_counts()

display(gender_distribution.to_frame("Patients"))

plt.figure(figsize=(7,5))
sns.barplot(x=gender_distribution.index, y=gender_distribution.values)
plt.title("Gender Distribution of Patients")
plt.xlabel("Sex")
plt.ylabel("Number of Patients")
plt.show()


## Basic 3 — What is the average resting blood pressure of patients?

In [ ]:
avg_trestbps = cleaned_data["trestbps"].mean()
print(f"Average resting blood pressure: {avg_trestbps:.2f} mm Hg")


## Basic 4 — How many patients have fasting blood sugar levels higher than 120 mg/dl?

In [ ]:
high_fbs_count = cleaned_data["fbs"].sum()
high_fbs_pct = cleaned_data["fbs"].mean() * 100

print("Patients with fasting blood sugar > 120 mg/dl:", int(high_fbs_count))
print(f"Percentage of patients: {high_fbs_pct:.2f}%")


## Basic 5 — What are the different types of chest pain recorded?

In [ ]:
cp_values = sorted(cleaned_data["cp"].unique())
print("Chest pain types recorded:", cp_values)

cp_distribution = cleaned_data["cp"].value_counts().sort_index()
display(cp_distribution.to_frame("Patients"))


## Basic 6 — What is the maximum heart rate achieved by patients?

In [ ]:
max_thalach = cleaned_data["thalach"].max()
print(f"Maximum heart rate achieved: {max_thalach} bpm")


## Basic 7 — What percentage of patients experience exercise-induced angina?

In [ ]:
exang_pct = cleaned_data["exang"].mean() * 100
print(f"Patients experiencing exercise-induced angina: {exang_pct:.2f}%")


## Basic 8 — What is the average cholesterol level in the dataset?

In [ ]:
avg_chol = cleaned_data["chol"].mean()
print(f"Average cholesterol level: {avg_chol:.2f} mg/dl")


## Basic 9 — How many patients have a resting ECG result of 2?

In [ ]:
restecg_2_count = (cleaned_data["restecg"] == 2).sum()
print("Patients with resting ECG result of 2:", int(restecg_2_count))


## Basic 10 — What is the distribution of the number of major vessels colored by fluoroscopy?

In [ ]:
ca_distribution = cleaned_data["ca"].value_counts().sort_index()

display(ca_distribution.to_frame("Patients"))

plt.figure(figsize=(8,5))
sns.barplot(x=ca_distribution.index.astype(str), y=ca_distribution.values)
plt.title("Distribution of Major Vessels Colored by Fluoroscopy")
plt.xlabel("Number of Major Vessels (ca)")
plt.ylabel("Number of Patients")
plt.show()


# MEDIUM LEVEL QUESTIONS

All 10 Medium questions from the supplied case study are answered below.

## Medium 1 — What is the correlation between age and cholesterol levels?

In [ ]:
age_chol_corr = cleaned_data[["age", "chol"]].corr()

display(age_chol_corr)

print(f"Pearson correlation between age and cholesterol: {cleaned_data['age'].corr(cleaned_data['chol']):.3f}")

plt.figure(figsize=(7,5))
sns.scatterplot(data=cleaned_data, x="age", y="chol", hue="target", alpha=0.65)
plt.title("Age vs Cholesterol")
plt.xlabel("Age")
plt.ylabel("Cholesterol (mg/dl)")
plt.show()


## Medium 2 — What is the distribution of chest pain types across different age groups?

In [ ]:
age_cp = cleaned_data.groupby("age")["cp"].value_counts().unstack(fill_value=0)

display(age_cp.head(20))

# Use age bands for a clearer overall comparison
bins = [0, 39, 49, 59, 69, 100]
labels = ["<40", "40–49", "50–59", "60–69", "70+"]

temp = cleaned_data.copy()
temp["age_group"] = pd.cut(temp["age"], bins=bins, labels=labels)

age_group_cp = pd.crosstab(temp["age_group"], temp["cp"])
display(age_group_cp)

age_group_cp.plot(kind="bar", stacked=True, figsize=(10,6))
plt.title("Chest Pain Type Distribution Across Age Groups")
plt.xlabel("Age Group")
plt.ylabel("Number of Patients")
plt.xticks(rotation=0)
plt.legend(title="Chest Pain Type")
plt.show()


## Medium 3 — How does maximum heart rate vary with exercise-induced angina?

In [ ]:
thalach_exang = cleaned_data.groupby("exang")["thalach"].mean()

display(thalach_exang.to_frame("Average Maximum Heart Rate"))

print("\n0 = No exercise-induced angina; 1 = Yes")
print(f"Without angina: {thalach_exang.loc[0]:.2f} bpm")
print(f"With angina: {thalach_exang.loc[1]:.2f} bpm")

plt.figure(figsize=(7,5))
sns.barplot(x=thalach_exang.index.map({0:"No",1:"Yes"}), y=thalach_exang.values)
plt.title("Average Maximum Heart Rate by Exercise-Induced Angina")
plt.xlabel("Exercise-Induced Angina")
plt.ylabel("Average Maximum Heart Rate (bpm)")
plt.show()


## Medium 4 — Is there a significant difference in resting blood pressure between male and female patients?

In [ ]:
male_bp = cleaned_data.loc[cleaned_data["sex"] == 1, "trestbps"]
female_bp = cleaned_data.loc[cleaned_data["sex"] == 0, "trestbps"]

male_mean = male_bp.mean()
female_mean = female_bp.mean()

t_stat, p_value = ttest_ind(male_bp, female_bp, equal_var=False)

print(f"Average resting BP — Male: {male_mean:.2f} mm Hg")
print(f"Average resting BP — Female: {female_mean:.2f} mm Hg")
print(f"Welch t-test statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("Result: The difference is statistically significant at the 5% level.")
else:
    print("Result: The difference is not statistically significant at the 5% level.")


## Medium 5 — What is the relationship between fasting blood sugar levels and the presence of heart disease?

In [ ]:
fbs_target = pd.crosstab(
    cleaned_data["fbs"],
    cleaned_data["target"],
    normalize="index"
) * 100

fbs_counts = pd.crosstab(cleaned_data["fbs"], cleaned_data["target"])

print("Counts:")
display(fbs_counts)

print("Row percentages:")
display(fbs_target)

fbs_counts.plot(kind="bar", stacked=True, figsize=(8,5))
plt.title("Fasting Blood Sugar vs Heart Disease")
plt.xlabel("Fasting Blood Sugar > 120 mg/dl (0 = No, 1 = Yes)")
plt.ylabel("Number of Patients")
plt.xticks(rotation=0)
plt.legend(title="Heart Disease")
plt.show()


## Medium 6 — How does the number of major vessels (ca) affect the target variable?

In [ ]:
ca_target = pd.crosstab(
    cleaned_data["ca"],
    cleaned_data["target"]
)

display(ca_target)

ca_target.plot(kind="bar", stacked=True, figsize=(9,5))
plt.title("Major Vessels (ca) vs Heart Disease")
plt.xlabel("Number of Major Vessels")
plt.ylabel("Number of Patients")
plt.xticks(rotation=0)
plt.legend(title="Heart Disease")
plt.show()

ca_rate = cleaned_data.groupby("ca")["target"].mean().mul(100)
display(ca_rate.to_frame("Heart Disease Rate (%)"))


## Medium 7 — What is the average oldpeak value for patients with different types of chest pain?

In [ ]:
oldpeak_cp = cleaned_data.groupby("cp")["oldpeak"].mean()

display(oldpeak_cp.to_frame("Average Oldpeak"))

plt.figure(figsize=(8,5))
sns.barplot(x=oldpeak_cp.index.astype(str), y=oldpeak_cp.values)
plt.title("Average Oldpeak by Chest Pain Type")
plt.xlabel("Chest Pain Type")
plt.ylabel("Average Oldpeak")
plt.show()


## Medium 8 — Analyze the distribution of thalassemia types among patients with heart disease.

In [ ]:
thal_target = pd.crosstab(
    cleaned_data["thal"],
    cleaned_data["target"]
)

display(thal_target)

heart_patients_thal = cleaned_data.loc[cleaned_data["target"] == 1, "thal"].value_counts().sort_index()
display(heart_patients_thal.to_frame("Heart Disease Patients"))

thal_target.plot(kind="bar", stacked=True, figsize=(9,5))
plt.title("Thalassemia Type vs Heart Disease")
plt.xlabel("Thalassemia Type")
plt.ylabel("Number of Patients")
plt.xticks(rotation=0)
plt.legend(title="Heart Disease")
plt.show()


## Medium 9 — What are the most common combinations of risk factors in patients with heart disease?

In [ ]:
heart_data = cleaned_data[cleaned_data["target"] == 1].copy()

risk_combinations = (
    heart_data.groupby(["cp", "fbs", "exang", "thal"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(risk_combinations.head(15))


## Medium 10 — Pairwise comparison of clinical measurements for patients with and without heart disease

In [ ]:
clinical_compare = (
    cleaned_data.groupby("target")[["age", "trestbps", "chol", "thalach", "oldpeak"]]
    .agg(["mean", "median", "std"])
)

display(clinical_compare)

# Box plots for key continuous measurements
plot_data = cleaned_data[["target", "age", "trestbps", "chol", "thalach", "oldpeak"]].copy()
plot_data["target_label"] = plot_data["target"].map({0:"No Heart Disease", 1:"Heart Disease"})

for col in ["age", "trestbps", "chol", "thalach", "oldpeak"]:
    plt.figure(figsize=(7,5))
    sns.boxplot(data=plot_data, x="target_label", y=col)
    plt.title(f"{col} by Heart Disease Status")
    plt.xlabel("Heart Disease Status")
    plt.ylabel(col)
    plt.xticks(rotation=0)
    plt.show()


# ADVANCED LEVEL QUESTIONS

The document asks for five advanced analyses. The final item is handled carefully because the supplied dataset does not contain a time-to-event or survival outcome; therefore a true survival-rate analysis cannot be performed from these columns alone.

## Advanced 1 — Effect of Combining Age, Cholesterol, and Blood Pressure

In [ ]:
pair_cols = ["age", "chol", "trestbps", "target"]

sns.pairplot(
    cleaned_data[pair_cols],
    hue="target",
    diag_kind="hist",
    plot_kws={"alpha": 0.6}
)
plt.suptitle("Combined Relationship of Age, Cholesterol and Blood Pressure with Heart Disease", y=1.02)
plt.show()

# Group patients into a simple combined risk-factor profile for descriptive analysis
temp = cleaned_data.copy()

age_high = temp["age"] >= temp["age"].median()
chol_high = temp["chol"] >= temp["chol"].median()
bp_high = temp["trestbps"] >= temp["trestbps"].median()

temp["combined_factor_count"] = age_high.astype(int) + chol_high.astype(int) + bp_high.astype(int)

combined_risk = (
    temp.groupby("combined_factor_count")["target"]
    .agg(["count", "mean"])
)

combined_risk["heart_disease_rate_pct"] = combined_risk["mean"] * 100
display(combined_risk)


## Advanced 2 — Which Clinical Measurement Has the Strongest Correlation with Heart Disease?

In [ ]:
corr_with_target = (
    cleaned_data.corr(numeric_only=True)["target"]
    .sort_values(ascending=False)
)

display(corr_with_target.to_frame("Correlation with Target"))

strongest_positive = corr_with_target.drop("target").idxmax()
strongest_negative = corr_with_target.drop("target").idxmin()

print("Strongest positive correlation with heart disease:", strongest_positive,
      f"({corr_with_target[strongest_positive]:.3f})")
print("Strongest negative correlation with heart disease:", strongest_negative,
      f"({corr_with_target[strongest_negative]:.3f})")

plt.figure(figsize=(8,7))
sns.heatmap(cleaned_data.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


## Advanced 3 — Logistic Regression to Predict Heart Disease Using Available Features

In [ ]:
features = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal"
]

X = cleaned_data[features]
y = cleaned_data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000))
])

logistic_pipeline.fit(X_train, y_train)

pred = logistic_pipeline.predict(X_test)
prob = logistic_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, pred), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))
print("\nClassification Report:")
print(classification_report(y_test, pred))

print("Confusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, pred),
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
))

coefficients = logistic_pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": coefficients,
    "Odds Ratio": np.exp(coefficients)
}).sort_values("Coefficient", ascending=False)

display(coef_df)

plt.figure(figsize=(10,7))
sns.barplot(data=coef_df, x="Coefficient", y="Feature")
plt.axvline(0, linewidth=1)
plt.title("Logistic Regression Coefficients")
plt.xlabel("Standardized Coefficient")
plt.ylabel("Feature")
plt.show()


## Advanced 4 — How do slope values vary with different chest pain types?

In [ ]:
slope_cp = pd.crosstab(
    cleaned_data["cp"],
    cleaned_data["slope"]
)

display(slope_cp)

slope_cp.plot(kind="bar", stacked=True, figsize=(9,5))
plt.title("Slope Distribution by Chest Pain Type")
plt.xlabel("Chest Pain Type")
plt.ylabel("Number of Patients")
plt.xticks(rotation=0)
plt.legend(title="Slope")
plt.show()

print("Average slope by chest pain type:")
display(cleaned_data.groupby("cp")["slope"].mean().to_frame("Average Slope"))


## Advanced 5 — Thalassemia and 'Survival Rates' Over a Period

In [ ]:
print("The supplied dataset does not contain a survival-time, follow-up-duration, or death/event date column.")
print("Therefore, a true survival-rate-over-time analysis cannot be validly calculated from this dataset.")

print("\nAvailable descriptive alternative:")
thal_target_age = (
    cleaned_data.groupby(["thal", "target"])["age"]
    .agg(["count", "mean"])
    .reset_index()
)

display(thal_target_age)

plt.figure(figsize=(9,5))
sns.lineplot(
    data=cleaned_data,
    x="age",
    y="thal",
    hue="target",
    marker="o",
    estimator="mean",
    errorbar=None
)
plt.title("Average Thalassemia Code by Age and Heart Disease Status")
plt.xlabel("Age")
plt.ylabel("Thalassemia Type")
plt.show()

print("\nInterpretation: this is a descriptive age/thal/diagnosis view, NOT a survival analysis.")


# Key Findings Summary

Run this cell after completing the analysis to produce a compact set of headline results.

In [ ]:
# Core results
heart_disease_rate = cleaned_data["target"].mean() * 100
male_pct = (cleaned_data["sex"] == 1).mean() * 100
female_pct = (cleaned_data["sex"] == 0).mean() * 100

top_cp = cleaned_data["cp"].value_counts().idxmax()
top_ca = cleaned_data["ca"].value_counts().idxmax()
top_thal = cleaned_data["thal"].value_counts().idxmax()

strongest_corr_feature = (
    corr_with_target.drop("target").abs().idxmax()
)
strongest_corr_value = corr_with_target[strongest_corr_feature]

print("===== PULSE OF PREVENTION — KEY RESULTS =====")
print(f"Total patients: {len(cleaned_data):,}")
print(f"Average age: {avg_age:.2f} years")
print(f"Male patients: {male_pct:.2f}%")
print(f"Female patients: {female_pct:.2f}%")
print(f"Average resting blood pressure: {avg_trestbps:.2f} mm Hg")
print(f"Average cholesterol: {avg_chol:.2f} mg/dl")
print(f"Heart disease prevalence in dataset: {heart_disease_rate:.2f}%")
print(f"Patients with fasting blood sugar >120 mg/dl: {high_fbs_count:,} ({high_fbs_pct:.2f}%)")
print(f"Exercise-induced angina: {exang_pct:.2f}%")
print(f"Maximum heart rate achieved: {max_thalach} bpm")
print(f"Most common chest pain type: {top_cp}")
print(f"Most common ca value: {top_ca}")
print(f"Most common thal value: {top_thal}")
print(f"Clinical variable with strongest absolute correlation with target: {strongest_corr_feature} ({strongest_corr_value:.3f})")


# High-Risk Patient Profile — Descriptive

This section creates a **data-derived profile**, not a medical diagnosis.

In [ ]:
# Compare selected variables between patients with and without heart disease
profile = cleaned_data.groupby("target")[
    ["age", "trestbps", "chol", "thalach", "oldpeak", "ca", "exang"]
].mean()

profile.index = ["No Heart Disease", "Heart Disease"]
display(profile)

print("\nImportant: These are group-level patterns in this dataset and should not be interpreted as individual medical diagnoses.")


# Business / Healthcare Recommendations

Use the actual results from the notebook when writing the final case-study conclusion.

### 1. Risk-factor monitoring
Prioritize monitoring of the clinical variables that show the strongest association with the target in the correlation and logistic-regression analyses.

### 2. High-risk profile identification
Use combinations of age, blood pressure, cholesterol, chest pain, exercise-induced angina, and other available clinical indicators to support risk-stratification workflows.

### 3. Preventive programs
Develop targeted preventive initiatives around the strongest observed risk patterns, while avoiding one-variable conclusions.

### 4. Clinical resource allocation
Use observed patterns in chest pain, major vessels, thalassemia type, and exercise response to help prioritize further clinical review.

### 5. Further research
The dataset is cross-sectional and does not include survival time or follow-up outcomes. Future studies should add longitudinal follow-up, treatment, and actual patient outcomes before making survival or causal claims.


## Final Conclusion

In [ ]:
print(
    "The analysis provides a structured view of patient demographics, clinical measurements, "
    "heart-disease patterns, and predictive relationships. These findings can support "
    "data-driven risk profiling and preventive planning, while clinical decisions should "
    "always be validated by qualified healthcare professionals."
)


## Download the Cleaned Dataset

In [ ]:
from google.colab import files
files.download(cleaned_file)
